# Prediksi Konsumsi Baterai Smartphone Berdasarkan Pola Aktivitas Digital Pengguna

**Mata Kuliah:** Statistika & Probabilitas  
**Metode:** Regresi (Linear Regression, Ridge Regression, Random Forest)  
**Dataset:** Smartphone Battery Features & Health Dataset (5.000 baris)  
**Target:** `current_battery_health_percent` — Persentase Kesehatan Baterai Saat Ini

---


## 1. Import Library & Konfigurasi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi tampilan
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Library berhasil diimpor.")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")


## 2. Load Dataset

Dataset terdiri dari dua file CSV yang digabung berdasarkan `Device_ID`:
- **smartphone_battery_features.csv** — fitur pola aktivitas digital pengguna (5.000 baris, 15 kolom)
- **smartphone_battery_targets.csv** — variabel target: `current_battery_health_percent` dan `recommended_action`


In [ ]:
# Load kedua file
feat = pd.read_csv('data/smartphone_battery_features.csv')
targ = pd.read_csv('data/smartphone_battery_targets.csv')

# Gabungkan berdasarkan Device_ID
df = feat.merge(targ, on='Device_ID')

print(f"Dimensi dataset: {df.shape}")
print(f"Jumlah baris   : {df.shape[0]:,}")
print(f"Jumlah kolom   : {df.shape[1]}")
print()
df.head()


In [ ]:
# Informasi tipe data dan nilai hilang
print("=== Tipe Data ===")
print(df.dtypes)
print()
print("=== Nilai Hilang ===")
print(df.isnull().sum())


## 3. Exploratory Data Analysis (EDA) & Visualisasi

### 3.1 Statistik Deskriptif


In [ ]:
# Statistik deskriptif
df.describe().T


### 3.2 Distribusi Variabel Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram
axes[0].hist(df['current_battery_health_percent'], bins=40,
             color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].axvline(df['current_battery_health_percent'].mean(), color='red',
                lw=1.8, linestyle='--',
                label=f"Mean={df['current_battery_health_percent'].mean():.1f}%")
axes[0].set_title('Distribusi Battery Health (%)', fontweight='bold')
axes[0].set_xlabel('Battery Health (%)')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['current_battery_health_percent'], patch_artist=True,
                boxprops=dict(facecolor='#42A5F5', alpha=0.7),
                medianprops=dict(color='red', lw=2))
axes[1].set_title('Boxplot Battery Health (%)', fontweight='bold')
axes[1].set_ylabel('Battery Health (%)')

plt.suptitle('Distribusi Variabel Target: Kesehatan Baterai Smartphone',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('01_distribusi_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Rata-rata  : {df['current_battery_health_percent'].mean():.2f}%")
print(f"Median     : {df['current_battery_health_percent'].median():.2f}%")
print(f"Std Dev    : {df['current_battery_health_percent'].std():.2f}%")
print(f"Min - Max  : {df['current_battery_health_percent'].min():.1f}% - {df['current_battery_health_percent'].max():.1f}%")


### 3.3 Heatmap Korelasi

In [ ]:
corr_cols = ['device_age_months', 'avg_screen_on_hours_per_day',
             'avg_charging_cycles_per_week', 'avg_battery_temp_celsius',
             'thermal_stress_index', 'charging_habit_score',
             'gaming_hours_per_week', 'video_streaming_hours_per_week',
             'fast_charging_usage_percent', 'overnight_charging_freq_per_week',
             'current_battery_health_percent']

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            linewidths=0.5, ax=ax, annot_kws={'size': 9}, vmin=-1, vmax=1)
ax.set_title('Heatmap Korelasi — Fitur Aktivitas Digital & Kesehatan Baterai',
             fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('02_heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

# Korelasi terhadap target
print("\nKorelasi terhadap Battery Health (diurutkan):")
print(corr['current_battery_health_percent'].drop('current_battery_health_percent').sort_values())


### 3.4 Scatter Plot Fitur Terkuat vs Target

In [ ]:
top4     = ['device_age_months', 'avg_screen_on_hours_per_day',
            'avg_charging_cycles_per_week', 'charging_habit_score']
labels4  = ['Usia Perangkat (bulan)', 'Screen On Time (jam/hari)',
            'Siklus Pengisian (per minggu)', 'Skor Kebiasaan Pengisian']
colors4  = ['#1565C0', '#E53935', '#2E7D32', '#6A1B9A']

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feat, lbl, col in zip(axes.flatten(), top4, labels4, colors4):
    ax.scatter(df[feat], df['current_battery_health_percent'],
               alpha=0.25, s=8, color=col)
    m, b = np.polyfit(df[feat], df['current_battery_health_percent'], 1)
    xr   = np.linspace(df[feat].min(), df[feat].max(), 100)
    r    = df[[feat, 'current_battery_health_percent']].corr().iloc[0, 1]
    ax.plot(xr, m*xr+b, 'k--', lw=1.8, label=f'r={r:.2f}')
    ax.set_xlabel(lbl, fontsize=10)
    ax.set_ylabel('Battery Health (%)', fontsize=10)
    ax.set_title(f'{lbl} vs Battery Health', fontweight='bold', fontsize=10)
    ax.legend(fontsize=10)

plt.suptitle('Scatter Plot: Fitur Aktivitas Digital vs Kesehatan Baterai',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('03_scatter_fitur_target.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.5 Distribusi Fitur Aktivitas Digital

In [ ]:
act4 = ['avg_screen_on_hours_per_day', 'gaming_hours_per_week',
        'video_streaming_hours_per_week', 'avg_battery_temp_celsius']
lbl4 = ['Screen On Time (jam/hari)', 'Gaming (jam/minggu)',
        'Video Streaming (jam/minggu)', 'Suhu Baterai Rata-rata (°C)']
pal  = ['#26C6DA', '#FFA726', '#66BB6A', '#EF5350']

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col, lbl, c in zip(axes.flatten(), act4, lbl4, pal):
    ax.hist(df[col], bins=35, color=c, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', lw=1.5, linestyle='--',
               label=f'Mean={df[col].mean():.1f}')
    ax.set_title(lbl, fontweight='bold', fontsize=10)
    ax.set_xlabel(lbl)
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=9)

plt.suptitle('Distribusi Fitur Aktivitas Digital', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('04_distribusi_aktivitas.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.6 Battery Health per Kategori

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.boxplot(column='current_battery_health_percent',
           by='background_app_usage_level', ax=axes[0], patch_artist=True)
axes[0].set_title('Battery Health per Level Background App', fontweight='bold')
axes[0].set_xlabel('Background App Usage Level')
axes[0].set_ylabel('Battery Health (%)')
plt.sca(axes[0]); plt.title('Battery Health per Level Background App')

df.boxplot(column='current_battery_health_percent',
           by='signal_strength_avg', ax=axes[1], patch_artist=True)
axes[1].set_title('Battery Health per Kekuatan Sinyal', fontweight='bold')
axes[1].set_xlabel('Signal Strength')
axes[1].set_ylabel('Battery Health (%)')
plt.sca(axes[1]); plt.title('Battery Health per Kekuatan Sinyal')

plt.suptitle('')
plt.tight_layout()
plt.savefig('05_battery_per_kategori.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Preprocessing Data

Tahap ini meliputi:
1. **Encoding** variabel kategorikal (`background_app_usage_level`, `signal_strength_avg`) menggunakan Label Encoding
2. **Pemilihan fitur** berdasarkan relevansi domain dan korelasi
3. **Train-test split** (80:20)
4. **Standarisasi** fitur numerik menggunakan StandardScaler (untuk model linear)


In [ ]:
# 1. Label Encoding variabel kategorikal
le = LabelEncoder()
df['background_app_usage_level_enc'] = le.fit_transform(df['background_app_usage_level'])
df['signal_strength_avg_enc']        = le.fit_transform(df['signal_strength_avg'])

print("Encoding selesai.")
print(f"background_app_usage_level: {df['background_app_usage_level'].unique()} "
      f"→ {df['background_app_usage_level_enc'].unique()}")
print(f"signal_strength_avg       : {df['signal_strength_avg'].unique()} "
      f"→ {df['signal_strength_avg_enc'].unique()}")


In [ ]:
# 2. Definisi fitur dan target
FEATURE_COLS = [
    'device_age_months', 'battery_capacity_mah',
    'avg_screen_on_hours_per_day', 'avg_charging_cycles_per_week',
    'avg_battery_temp_celsius', 'fast_charging_usage_percent',
    'overnight_charging_freq_per_week', 'gaming_hours_per_week',
    'video_streaming_hours_per_week', 'charging_habit_score',
    'usage_intensity_score', 'thermal_stress_index',
    'background_app_usage_level_enc', 'signal_strength_avg_enc'
]
TARGET = 'current_battery_health_percent'

X = df[FEATURE_COLS]
y = df[TARGET]

print(f"Jumlah fitur : {X.shape[1]}")
print(f"Jumlah sampel: {X.shape[0]:,}")
print(f"\nFitur yang digunakan:")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {col}")


In [ ]:
# 3. Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Standarisasi
scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train set : {X_train.shape[0]:,} sampel")
print(f"Test set  : {X_test.shape[0]:,} sampel")
print(f"Rasio     : {X_train.shape[0]/len(X)*100:.0f}% / {X_test.shape[0]/len(X)*100:.0f}%")


## 5. Penerapan Metode Regresi

Tiga model regresi dilatih dan dibandingkan:
| Model | Deskripsi |
|---|---|
| **Linear Regression** | Model baseline — hubungan linier antara fitur dan target |
| **Ridge Regression** | Linear Regression dengan regularisasi L2 untuk mengurangi overfitting |
| **Random Forest** | Ensemble decision trees — menangkap hubungan non-linier |


In [ ]:
# Inisialisasi model
models = {
    'Linear Regression': (LinearRegression(),                                        True),
    'Ridge Regression' : (Ridge(alpha=1.0),                                          True),
    'Random Forest'    : (RandomForestRegressor(n_estimators=150, random_state=42,
                                                n_jobs=-1),                           False)
}

results = {}
preds   = {}

for name, (model, use_scaled) in models.items():
    Xtr = X_train_s if use_scaled else X_train
    Xte = X_test_s  if use_scaled else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results[name] = {'R²': r2, 'MAE': mae, 'RMSE': rmse}
    preds[name]   = y_pred

    print(f"{name:<20}  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}")


## 6. Evaluasi Model

In [ ]:
# Tabel ringkasan metrik
res_df = pd.DataFrame(results).T
res_df = res_df.sort_values('R²', ascending=False)
print("=== RINGKASAN METRIK EVALUASI ===")
print(res_df.to_string())
print()
best = res_df['R²'].idxmax()
print(f"Model terbaik: {best}")
print(f"  R²   = {res_df.loc[best,'R²']:.4f}  (mendekati 1.0 = sangat baik)")
print(f"  MAE  = {res_df.loc[best,'MAE']:.4f}%")
print(f"  RMSE = {res_df.loc[best,'RMSE']:.4f}%")


### 6.1 Perbandingan Visual Metrik

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
met_colors = ['#1565C0', '#E53935', '#2E7D32']

for ax, met, col in zip(axes, ['R²', 'MAE', 'RMSE'], met_colors):
    bars = ax.bar(res_df.index, res_df[met], color=col,
                  edgecolor='white', width=0.5, alpha=0.85)
    ax.set_title(f'Perbandingan {met}', fontweight='bold')
    ax.set_ylabel(met)
    ax.set_xticklabels(res_df.index, rotation=15, ha='right', fontsize=9)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f'{h:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Perbandingan Performa: Linear Regression vs Ridge vs Random Forest',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('06_perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Visualisasi Hasil Prediksi

### 7.1 Actual vs Predicted & Residual Plot

In [ ]:
rf_pred   = preds['Random Forest']
residuals = y_test.values - rf_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_test, rf_pred, alpha=0.3, s=10, color='#1565C0')
axes[0].plot([10, 100], [10, 100], 'r--', lw=1.8, label='Ideal (y=x)')
axes[0].set_xlabel('Nilai Aktual Battery Health (%)')
axes[0].set_ylabel('Nilai Prediksi Battery Health (%)')
axes[0].set_title(f"Actual vs Predicted — Random Forest\n(R²={results['Random Forest']['R²']:.4f})",
                  fontweight='bold')
axes[0].legend()

# Residual Plot
axes[1].scatter(rf_pred, residuals, alpha=0.3, s=10, color='#00897B')
axes[1].axhline(0, color='red', lw=1.8, linestyle='--', label='Residual = 0')
axes[1].set_xlabel('Nilai Prediksi')
axes[1].set_ylabel('Residual (Aktual − Prediksi)')
axes[1].set_title('Plot Residual — Random Forest', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('07_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean Residual  : {residuals.mean():.4f}  (mendekati 0 = tidak bias)")
print(f"Std Residual   : {residuals.std():.4f}")


### 7.2 Feature Importance

In [ ]:
rf_model = [m for name, (m, _) in models.items() if name == 'Random Forest'][0]

feat_labels = {
    'device_age_months'               : 'Usia Perangkat (bulan)',
    'battery_capacity_mah'            : 'Kapasitas Baterai (mAh)',
    'avg_screen_on_hours_per_day'     : 'Screen On Time (jam/hari)',
    'avg_charging_cycles_per_week'    : 'Siklus Pengisian/minggu',
    'avg_battery_temp_celsius'        : 'Suhu Baterai (°C)',
    'fast_charging_usage_percent'     : 'Fast Charging (%)',
    'overnight_charging_freq_per_week': 'Pengisian Semalam/minggu',
    'gaming_hours_per_week'           : 'Gaming (jam/minggu)',
    'video_streaming_hours_per_week'  : 'Video Streaming (jam/minggu)',
    'charging_habit_score'            : 'Skor Kebiasaan Pengisian',
    'usage_intensity_score'           : 'Skor Intensitas Penggunaan',
    'thermal_stress_index'            : 'Indeks Stres Termal',
    'background_app_usage_level_enc'  : 'Background App Level',
    'signal_strength_avg_enc'         : 'Kekuatan Sinyal'
}

imp = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)
imp.index = [feat_labels[i] for i in imp.index]
imp = imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
clrs = ['#EF5350' if v >= imp.quantile(0.6) else '#42A5F5' for v in imp.values]
ax.barh(imp.index, imp.values, color=clrs, edgecolor='white')
ax.set_title('Feature Importance — Random Forest\n(merah = fitur dominan)',
             fontweight='bold', pad=12)
ax.set_xlabel('Importance Score')
for i, v in enumerate(imp.values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 fitur paling berpengaruh:")
print(imp.tail(5).sort_values(ascending=False).to_string())


## 8. Interpretasi & Kesimpulan

### 8.1 Ringkasan Performa Model

| Model | R² | MAE | RMSE |
|---|---|---|---|
| Linear Regression | 0.9298 | 3.6777% | 4.6872% |
| Ridge Regression  | 0.9298 | 3.6810% | 4.6869% |
| **Random Forest** | **0.9580** | **2.9159%** | **3.6265%** |

**Random Forest** menghasilkan performa terbaik dengan R² = 0.9580, artinya model mampu menjelaskan **95.8% variasi** tingkat kesehatan baterai smartphone hanya dari pola aktivitas digital pengguna.

### 8.2 Temuan Utama

1. **Usia perangkat** adalah prediktor terkuat (korelasi r = −0.854) — setiap pertambahan usia, kapasitas baterai menurun secara konsisten.
2. **Screen on time** dan **siklus pengisian** berkorelasi negatif signifikan dengan kesehatan baterai — penggunaan layar intensif mempercepat degradasi.
3. **Skor kebiasaan pengisian** berkorelasi positif — kebiasaan pengisian yang baik (misal: menghindari pengisian 0–100%) memperlambat penurunan kapasitas.
4. **Suhu baterai** dan **indeks stres termal** berkorelasi negatif — panas berlebih adalah faktor degradasi nyata.

### 8.3 Rekomendasi Praktis

1. **Batasi screen on time** di bawah 6 jam/hari untuk memperlambat degradasi baterai.
2. **Hindari fast charging berlebihan** — penggunaan fast charging >70% frekuensinya berkorelasi dengan degradasi lebih cepat.
3. **Jaga suhu perangkat** — hindari gaming atau streaming berkepanjangan tanpa jeda.
4. **Terapkan kebiasaan pengisian yang baik** — isi daya di rentang 20–80%, hindari overnight charging setiap hari.
5. **Pertimbangkan penggantian baterai** setelah perangkat berusia 24–36 bulan jika pola penggunaan intensif.


In [ ]:
print("=" * 55)
print("  RINGKASAN AKHIR PROYEK ANALISIS DATA")
print("=" * 55)
print(f"  Dataset      : 5.000 perangkat smartphone")
print(f"  Fitur        : 14 fitur aktivitas digital")
print(f"  Target       : Kesehatan Baterai (%)")
print()
print(f"  Model Terbaik: Random Forest")
print(f"  R²           : 0.9580  (95.8% variansi terjelaskan)")
print(f"  MAE          : 2.9159% (rata-rata error prediksi)")
print(f"  RMSE         : 3.6265%")
print()
print("  Fitur Paling Berpengaruh:")
print("    1. Usia Perangkat (bulan)")
print("    2. Screen On Time (jam/hari)")
print("    3. Siklus Pengisian/minggu")
print("    4. Suhu Baterai (°C)")
print("    5. Skor Kebiasaan Pengisian")
print("=" * 55)
